# 🎯 Workshop Day 3: Evaluation & Optimization

**Agentic RAG Workshop** | Day 3 of 3

---

### 🎯 English
1. **English RAG** — English RAGAS (4 metrics)
2. **English Eval Dataset** — English Gemini
3. **English Agent** — Tool Selection, LLM-as-Judge
4. **English Pipeline** — A/B Experiment
5. **Prompt Engineering** — English Instruction English
6. **Capstone Project** — English 3 English ⭐

### 🗺️ English 3 English

```
Day 1 (Data)           Day 2 (Agent)          Day 3 (Evaluation)
─────────────          ─────────────          ─────────────────
Raw → Chunk →          Agent + Tool →         English (3.1-3.2)
Embed → VectorDB       RAG Agent →            English Agent (3.3)
  → Retrieve           Multi-Agent →          English (3.4-3.5)
                       Agentic RAG            Capstone ⭐ (3.6)
```

> 💡 English "English" — English English

## 📦 Section 0: English Dependencies

In [ ]:
%%time
!pip install -q ragas datasets google-adk google-genai \
    sentence-transformers qdrant-client langchain-text-splitters \
    rank_bm25 scikit-learn

import warnings
warnings.filterwarnings('ignore')
print('✅ English!')

In [ ]:
%%time
import os, json, hashlib, random, asyncio
import numpy as np

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ English API Key English Colab Secrets')
except Exception:
    api_key = input('🔑 English Gemini API Key: ')
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
try:
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    resp = client.models.generate_content(model='gemini-2.5-flash', contents='English OK')
    print(f'✅ API English: {(resp.text.strip() if resp.text else '(English output)')}')
except Exception as e:
    print(f'❌ API Error: {e}')

### English + VectorDB (re-use English Day 1-2)

In [ ]:
%%time
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')

documents = {
    'healthcare': 'English English AI English X-ray English English 30 English 5 English English Deep Learning English 95% English English',
    'banking': 'English English KADE AI Assistant English RAG English English 5 English 30 English English English English English',
    'education': 'English English RAG English-English English PDF English 500 English vector embeddings English multilingual model English English',
    'agriculture': 'English English AI English English English 92% English 40% English Computer Vision English CNN English train English 50000 English',
    'rag': 'English RAG English 3 English Retriever English VectorDB Reader English Generator English English hallucination English English LLM English',
    'embedding': 'Text Embedding English vector multilingual-e5-large English 100 English English vector 1024 English English vector English English Cosine Similarity English',
    'kmitl': 'English (KMITL) English English AI English Smart Campus English IoT sensor English Machine Learning English English 25% English NLP English English RAG English AI Tutor English',
    'nlp': 'NLP English English English Word Segmentation English PyThaiNLP English English English Tokenizer English',
    'vectordb': 'Qdrant English Vector Database English English ANN English vectors English Cosine Dot Product L2 Distance English payload filter English metadata English',
}

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
all_chunks, all_sources = [], []
for src, text in documents.items():
    for chunk in splitter.split_text(text):
        all_chunks.append(chunk)
        all_sources.append(src)

passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
qdrant.create_collection('thai_ai_kb',
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE))
points = [models.PointStruct(id=i, vector=embeddings[i].tolist(),
    payload={'text': all_chunks[i], 'source': all_sources[i]}) for i in range(len(all_chunks))]
qdrant.upsert('thai_ai_kb', points=points)

print(f'✅ Data ready: {len(all_chunks)} chunks, {len(documents)} sources')

# RAG function
def search_qdrant(query, top_k=3):
    q_vec = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points('thai_ai_kb', query=q_vec, limit=top_k).points
    return [{'text': r.payload['text'], 'source': r.payload['source'],
             'score': round(r.score, 4)} for r in results]

def rag_answer(question, top_k=3):
    contexts = search_qdrant(question, top_k)
    context_text = '\n'.join([c['text'] for c in contexts])
    prompt = f'"""English English English\n\nEnglish:\n{context_text}\n\nEnglish: {question}\nEnglish:"""'
    resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.3))
    return (resp.text.strip() if resp.text else '(English output)'), contexts

# Quick test
ans, ctx = rag_answer('AI English')
print(f'\n🧪 Test RAG: {ans[:80]}...')

---
## 📊 Section 3.1: RAGAS — English RAG

### RAGAS English?

**RAGAS (Retrieval-Augmented Generation Assessment Suite)** = framework English RAG

```
Input: question + answer + contexts (+ ground_truth)
  ↓ RAGAS English
Output: English 0.0 - 1.0 English metric
```

### 4 Metrics English

| Metric | English | English | English ground_truth? |
|--------|---------|------------|:------------------:|
| **Faithfulness** | English "English" English? (English?) | answer ↔ contexts | ❌ |
| **Answer Relevancy** | English? | answer ↔ question | ❌ |
| **Context Precision** | English chunk English? | contexts ↔ ground_truth | ✅ |
| **Context Recall** | English chunk English? | contexts ↔ ground_truth | ✅ |

### 🌡️ English

| English | English | English |
|:-----:|-------|----------|
| 0.8-1.0 | 🟢 English | ✅ English |
| 0.6-0.8 | 🟡 English | ⚠️ English |
| < 0.6 | 🔴 English | ❌ English |

In [ ]:
%%time
# ─── English Evaluation Dataset ───
# English: question, answer, contexts, ground_truth

eval_questions = [
    'AI English',
    'RAG English',
    'English AI English',
    'NLP English',
    'Qdrant English',
]

eval_ground_truths = [
    'English AI English X-ray English English 30 English 5 English English 95%',
    'RAG English 3 English Retriever English VectorDB, Reader English, Generator English English hallucination',
    'English KADE AI Assistant English RAG English English 5 English 30 English',
    'English English Word Segmentation English PyThaiNLP English English',
    'Qdrant English Vector Database English English ANN English English Cosine Dot Product L2',
]

# English RAG English answers + contexts
eval_answers = []
eval_contexts = []

print('🔄 English answers...')
for q in eval_questions:
    ans, ctxs = rag_answer(q)
    eval_answers.append(ans)
    eval_contexts.append([c['text'] for c in ctxs])
    print(f'  ✅ {q[:30]}... → {ans[:40]}...')

print(f'\n📊 Eval Dataset: {len(eval_questions)} questions')

In [ ]:
%%time
# ─── English RAGAS ───
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

eval_dataset = Dataset.from_dict({
    'question': eval_questions,
    'answer': eval_answers,
    'contexts': eval_contexts,
    'ground_truth': eval_ground_truths,
})

print('🔄 English RAGAS... (English 1-2 English)')
try:
    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    )
    print(f'\n{"="*60}')
    print(f'📊 RAGAS Evaluation Results')
    print(f'{"="*60}')
    for metric, score in result.items():
        level = '🟢' if score >= 0.8 else ('🟡' if score >= 0.6 else '🔴')
        print(f'  {level} {metric}: {score:.4f}')
except Exception as e:
    print(f'⚠️ RAGAS error: {e}')
    print('💡 English manual evaluation English (Section 3.3)')

### 💡 English
- **Faithfulness English** = Agent English English
- **Context Precision English** = English chunk English → English `chunk_size`, `top_k`
- **Context Recall English** = English → English `top_k` English embedding

> 🎯 English: English metric **≥ 0.7** English production

### 🧪 English 3.1
1. English 3 English + ground_truth
2. English RAGAS → metric English?
3. English

In [ ]:
# 🧪 English: English RAGAS
# 💡 Hint:
#   1. English question + ground_truth English list
#   2. English rag_answer() English answer + contexts
#   3. English Dataset.from_dict() English evaluate()


---
## 🔬 Section 3.2: English Evaluation Dataset English

### English?

- English ground_truth English → English (5 English = 30 English)
- English LLM English chunk → English 50 English 2 English

```
Chunk: "English AI English X-ray English 95%"
  ↓ Gemini English
Q: "English AI English X-ray?"
A: "English English 95%"
```

In [ ]:
%%time
# ─── Auto-generate Q&A English chunks ───
def generate_qa_from_chunk(chunk_text):
    prompt = f'''English English 1 English + English
English: {chunk_text}

English JSON: {{"question": "...", "answer": "..."}}
English English English'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.3, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return None

# English 8 chunks (English chunk English source)
auto_qa = []
seen_sources = set()
for i, chunk in enumerate(all_chunks):
    src = all_sources[i]
    if src in seen_sources: continue
    seen_sources.add(src)
    qa = generate_qa_from_chunk(chunk)
    if qa:
        qa['source'] = src
        auto_qa.append(qa)
        print(f'  ✅ [{src}] Q: {qa["question"][:50]}...')

print(f'\n📊 English {len(auto_qa)} Q&A pairs')

### 💡 English
- LLM English 10 English
- English **English** — English
- English **starting point** English

> 🎯 Production: English 100 English → English → English 50 English

### 🧪 English 3.2
English auto Q&A English chunks English English

In [ ]:
# 🧪 English
# 💡 Hint: English all_chunks English Q&A English generate_qa_from_chunk()


---
## 🧪 Section 3.3: English Agent

### English?

| English | English | English |
|--------|----------|--------|
| **Tool Selection** | English tool English? | English BMI → English calculate_bmi |
| **Response Quality** | English? | English LLM-as-Judge |
| **Edge Cases** | English? | English, English |
| **Guardrails** | English? | English |

In [ ]:
# ─── English Agent English ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

def search_knowledge(query: str) -> dict:
    """English AI English
    Args:
        query: English
    """
    results = search_qdrant(query, top_k=3)
    return {'status': 'success', 'results': results}

def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """English BMI
    Args:
        weight_kg: English
        height_cm: English
    """
    h = height_cm / 100
    bmi = weight_kg / (h ** 2)
    return {'bmi': round(bmi, 1)}

test_agent = Agent(
    name='test_agent', model='gemini-2.5-flash',
    instruction='''English AI English
    - English search_knowledge English AI English
    - English calculate_bmi English BMI
    - English English
    - English''',
    tools=[search_knowledge, calculate_bmi]
)

async def test_agent_call(agent, msg):
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session_id = f's_{id(agent)}_{id(msg)}'
    await runner.session_service.create_session(
        app_name=agent.name, user_id='tester', session_id=session_id
    )
    content = types.Content(role='user', parts=[types.Part.from_text(text=msg)])
    response, tools = '', []
    async for event in runner.run_async(user_id='tester', session_id=session_id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text: response = p.text
                if p.function_call: tools.append(p.function_call.name)
    return response, tools

print('✅ Test Agent English')

In [ ]:
# ─── Test Suite: Tool Selection ───
test_cases = [
    {'input': 'English 70 English 175 BMI?',      'expected_tool': 'calculate_bmi'},
    {'input': 'AI English',            'expected_tool': 'search_knowledge'},
    {'input': 'English',                     'expected_tool': None},
    {'input': 'Qdrant English',                 'expected_tool': 'search_knowledge'},
    {'input': 'English',            'expected_tool': None},
]

print('═' * 60)
print('🧪 Test Suite: Tool Selection')
print('═' * 60)

passed = 0
for tc in test_cases:
    resp, tools = await test_agent_call(test_agent, tc['input'])
    actual_tool = tools[0] if tools else None
    ok = actual_tool == tc['expected_tool']
    passed += ok
    status = '✅' if ok else '❌'
    print(f"  {status} '{tc['input'][:25]}...' → expected={tc['expected_tool']}, got={actual_tool}")

print(f'\n📊 Pass rate: {passed}/{len(test_cases)} ({passed/len(test_cases)*100:.0f}%)')

In [ ]:
%%time
# ─── LLM-as-Judge ───
def llm_judge(question, answer, max_score=5):
    prompt = f'''English 1-{max_score} English:
- English (English)
- English (English)
- English

English: {question}
English: {answer}

English JSON: {{"score": 1-{max_score}, "reason": "English"}}'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return {'score': 0, 'reason': 'error'}

# English
print('═' * 60)
print('🧪 LLM-as-Judge: English')
print('═' * 60)

judge_questions = [
    'AI English',
    'RAG English',
    'English AI English',
]

total_score = 0
for q in judge_questions:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    total_score += verdict['score']
    print(f"  {'⭐'*verdict['score']}{'☆'*(5-verdict['score'])} {verdict['score']}/5 | {q[:25]}... → {verdict['reason']}")

avg = total_score / len(judge_questions)
print(f'\n📊 Average: {avg:.1f}/5.0')

### 💡 English
- **Tool Selection test** = English Agent English tool English (deterministic)
- **LLM-as-Judge** = English LLM English (English)
- English → English "English" English "English"

> ⚠️ LLM-as-Judge English 100% English — English screening tool English human review

### 🧪 English 3.3
English test cases 5 English English edge cases (English, English)

In [ ]:
# 🧪 English
# 💡 Hint: English test_cases English English test_agent_call()


---
## ⚡ Section 3.4: English RAG Pipeline

### English?

| English | Parameter | English |
|---------|----------|----------|
| Chunking | `chunk_size`, `overlap` | Context Precision |
| Retrieval | `top_k`, `alpha` | Context Recall |
| Generation | `temperature`, prompt | Faithfulness, Relevancy |

### A/B Experiment
```
Config A: chunk=100, top_k=3
Config B: chunk=200, top_k=5  ← English?
Config C: chunk=300, top_k=3
```

In [ ]:
%%time
# ─── A/B Experiment ───
configs = [
    {'name': 'A: small chunks', 'chunk_size': 100, 'overlap': 20, 'top_k': 3},
    {'name': 'B: medium chunks', 'chunk_size': 200, 'overlap': 30, 'top_k': 5},
    {'name': 'C: large chunks', 'chunk_size': 300, 'overlap': 50, 'top_k': 3},
]

test_q = 'AI English'
test_gt = 'English AI English X-ray English 95%'

print('═' * 60)
print('🧪 A/B Experiment: English Config')
print('═' * 60)

for cfg in configs:
    sp = RecursiveCharacterTextSplitter(chunk_size=cfg['chunk_size'], chunk_overlap=cfg['overlap'])
    chunks = []
    for text in documents.values():
        chunks.extend(sp.split_text(text))
    
    # Embed + search
    vecs = embed_model.encode([f'passage: {c}' for c in chunks])
    q_vec = embed_model.encode(f'query: {test_q}')
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity([q_vec], vecs)[0]
    top_idx = np.argsort(sims)[-cfg['top_k']:][::-1]
    top_chunks = [chunks[i] for i in top_idx]
    top_scores = [sims[i] for i in top_idx]
    
    # Check if ground truth info is in retrieved chunks
    gt_found = any('English' in c or '95%' in c for c in top_chunks)
    
    print(f'\n📋 {cfg["name"]}: {len(chunks)} chunks')
    print(f'   Top scores: {[round(s,3) for s in top_scores]}')
    print(f'   Ground truth found: {"✅" if gt_found else "❌"}')
    print(f'   Best chunk: {top_chunks[0][:60]}...')

### 💡 English
- **chunk English** (100) → chunks English English
- **chunk English** (300) → English English noise English
- **top_k English** → recall English English precision English

> 🎯 English config English — English **English** English

### 🧪 English 3.4
English 3 configs → English config English

In [ ]:
# 🧪 English: English configs English
# 💡 Hint: English configs English list English


---
## ✍️ Section 3.5: Prompt Engineering English Agent

### English

| English | English | English Hallucination? |
|--------|---------|:-----------------:|
| **Role** | "English..." | ⭐ |
| **Few-shot** | English Q&A | ⭐⭐ |
| **Chain-of-Thought** | "English..." | ⭐⭐⭐ |
| **Guardrails** | "English English" | ⭐⭐⭐ |
| **Output Format** | "English bullet 3 English" | ⭐ |

In [ ]:
# ─── Before vs After Prompt ───
prompt_before = 'English'
prompt_after = '''English AI English English

English:
1. English search_knowledge English
2. English English
3. English English [healthcare] [banking]
4. English English English "English"
5. English English bullet points English 5 English'''

agent_v1 = Agent(name='v1', model='gemini-2.5-flash', instruction=prompt_before, tools=[search_knowledge])
agent_v2 = Agent(name='v2', model='gemini-2.5-flash', instruction=prompt_after, tools=[search_knowledge])

test_q = 'AI English'

print('═' * 60)
print('🧪 Before vs After Prompt')
print('═' * 60)

r1, _ = await test_agent_call(agent_v1, test_q)
r2, _ = await test_agent_call(agent_v2, test_q)

j1 = llm_judge(test_q, r1)
j2 = llm_judge(test_q, r2)

print(f'\n📋 V1 (instruction English): {j1["score"]}/5 — {j1["reason"]}')
print(f'📋 V2 (instruction English):   {j2["score"]}/5 — {j2["reason"]}')
print(f'\n--- V1 ---\n{r1[:200]}')
print(f'\n--- V2 ---\n{r2[:200]}')

### 💡 English
- Instruction English → English **English, English, English**
- Instruction English → English **English, English, English reference**
- **Guardrails** ("English") English hallucination English

> 🎯 Prompt Engineering = English **English English**

### 🧪 English 3.5
English instruction 3 English → English LLM-as-Judge → English

In [ ]:
# 🧪 English
# 💡 Hint: English agent_v3 English instruction English English


---
## 🚀 Section 3.6: Capstone Project ⭐

### English 3 English

```
Day 1: English → Chunk → Embed → VectorDB
Day 2: English Agent → Tools → RAG Agent → Multi-Agent
Day 3: English RAGAS → English → English → English
```

### English
1. **English RAG Pipeline** English
2. **English** English RAGAS + LLM-as-Judge
3. **English** English (chunk_size, prompt, top_k)
4. **English Before/After**

In [ ]:
# ─── Capstone: Before → Ater ───
print('═' * 60)
print('🚀 Capstone: English RAG Pipeline')
print('═' * 60)

# Step 1: English baseline
print('\n📊 Step 1: Baseline')
baseline_scores = []
for q in eval_questions[:3]:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    baseline_scores.append(verdict['score'])
baseline_avg = sum(baseline_scores) / len(baseline_scores)
print(f'   Baseline Score: {baseline_avg:.1f}/5.0')

# Step 2: English (English: prompt English)
print('\n📊 Step 2: After Optimization')
improved_scores = []
for q in eval_questions[:3]:
    r, _ = await test_agent_call(agent_v2, q)
    verdict = llm_judge(q, r)
    improved_scores.append(verdict['score'])
improved_avg = sum(improved_scores) / len(improved_scores)
print(f'   Improved Score: {improved_avg:.1f}/5.0')

# Step 3: English
print(f'\n{"═"*60}')
print(f'📊 English Before → After')
print(f'{"═"*60}')
print(f'  Baseline:  {"⭐" * round(baseline_avg)} {baseline_avg:.1f}/5.0')
print(f'  Improved:  {"⭐" * round(improved_avg)} {improved_avg:.1f}/5.0')
diff = improved_avg - baseline_avg
if diff > 0:
    print(f'  📈 +{diff:.1f} English!')
elif diff < 0:
    print(f'  📉 {diff:.1f} English — English baseline')
else:
    print(f'  ➡️ English — English')

### 💡 English
- **Prompt Engineering** English
- **Chunk size** English — English
- **English-English** English — English

> 🎯 "What gets measured gets improved" — English

### 🧪 English 3.6
1. English 2 English (prompt + chunk_size English top_k)
2. English Before/After
3. English

In [ ]:
# 🧪 Capstone: English
# 💡 Hint:
#   1. English chunk_size → re-embed → re-search → English
#   2. English instruction → English agent English → English
#   3. English top_k → rag_answer( ,top_k=5) → English


---
## 🎯 English: English 3 English

| Day | English | English |
|-----|--------|-------------|
| **Day 1** | Data Engineering Pipeline | Qdrant, E5-large, BM25 |
| **Day 2** | Agent Building | Google ADK, Tools, Multi-Agent |
| **Day 3** | Evaluation & Optimization | RAGAS, LLM-as-Judge |

### Skills English
- ✅ English RAG Pipeline English
- ✅ English Agent English
- ✅ English + English
- ✅ English

### 🛣️ English?
- **Production**: Deploy English Cloud (GCP, AWS) + Database session
- **Advanced**: Fine-tune embedding English domain English
- **Scale**: Qdrant Cloud + horizontal scaling
- **Monitor**: Log + alert English

---

**🙏 English 3 English!**

> "The best way to learn AI is to build with AI."